# Embeddings de modelos de lenguaje de ADN sobre los promotores del MPRA

Extrae representaciones congeladas de tres modelos preentrenados y las evalúa con el mismo
split por cromosoma y el mismo ridge que las bases (`evaluar.py`).

| Modelo | Arquitectura | Tokens | Capas | Dimensión |
|---|---|---|---|---|
| DNABERT-2 (117M) | BERT | BPE (longitud variable) | 12 | 768 |
| Nucleotide Transformer v2 (100M, multiespecie) | BERT | 6-meros | 22 | 512 |
| HyenaDNA small-32k | Hyena (causal) | 1 nucleótido | 4 | 256 |

**Antes de correr:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`. Subí a una carpeta de
Drive los dos archivos `embeddings_adn_dataset_v1.tsv` (de `TesisDoc/data/processed/embeddings_adn/`) y `evaluar.py`
(de `TesisDoc/R/71_exploratorio_embeddings_adn/`), y poné la ruta en `CARPETA`.

Para cada modelo se guardan 4 capas (a 1/4, 1/2, 3/4 y al final de la red) y dos formas de resumir
la secuencia en un vector: **media** (promedio de todos los tokens) y **ventanas** (promedio dentro de
−236 a −61, −60 a −11 y −10 a +15 respecto del TSS, concatenados; conserva algo de posición).
La capa se elige dentro del entrenamiento de cada fold, nunca mirando el test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CARPETA = '/content/drive/MyDrive/embeddings_adn'   # <- cambiar si hace falta

In [ ]:
# El código propio de Nucleotide Transformer y DNABERT-2 (trust_remote_code) depende de
# internals de transformers que las versiones nuevas eliminaron (find_pruneable_heads_and_indices,
# config.is_decoder, ...). Se fija una versión de fines de 2024 para los tres modelos.
# DNABERT-2 trae además una atención en triton que no compila con las versiones actuales;
# sin triton cae a la implementación estándar de PyTorch.
!pip -q install einops "transformers==4.46.3"
!pip -q uninstall -y triton

**Después de la celda anterior: `Entorno de ejecución → Reiniciar sesión`**, y seguir desde la celda
siguiente (volver a correr la de Drive si hace falta). Comprobar con
`import transformers; transformers.__version__` que diga 4.46.3.

In [ ]:
import sys, time, numpy as np, pandas as pd, torch
sys.path.insert(0, CARPETA)
import evaluar as ev
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM, AutoConfig

df = ev.cargar(f'{CARPETA}/embeddings_adn_dataset_v1.tsv')
seqs = df.seq.tolist()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(len(df), 'promotores |', dev)

## Extracción

`spans_nucleotidicos` arma, para cada token que corresponde a secuencia (descarta `[CLS]`, `[SEP]`, etc.),
el intervalo de pb que cubre. Sirve igual para un token por nucleótido (HyenaDNA), 6-meros (NT) o BPE
(DNABERT-2), y es lo que permite el pooling por ventanas.

In [ ]:
VENT = list(ev.VENTANAS.values())

def spans_nucleotidicos(tok, ids):
    toks = tok.convert_ids_to_tokens(ids)
    keep, spans, pos = [], [], 0
    for i, t in enumerate(toks):
        t = t.replace('##', '')
        if t and set(t) <= set('ACGT'):
            keep.append(i); spans.append((pos, pos + len(t))); pos += len(t)
    assert pos == 252, f'los tokens cubren {pos} pb'
    return np.array(keep), np.array(spans)

def resumir(h, spans):
    # h: tokens x d (solo tokens de secuencia). Devuelve (media, ventanas concatenadas)
    centro = spans.mean(1)
    vent = [h[(centro >= a) & (centro < b)].mean(0) for a, b in VENT]
    return h.mean(0), np.concatenate(vent)

def capas_a_guardar(n_capas):
    # hidden_states[0] es la salida de los embeddings; 1..n son los bloques
    return sorted({max(1, round(f * n_capas)) for f in (0.25, 0.5, 0.75, 1.0)})

def extraer(nombre, tok, forward, n_capas, batch):
    capas = capas_a_guardar(n_capas)
    media = {c: [] for c in capas}; vent = {c: [] for c in capas}
    t0 = time.time()
    for i in range(0, len(seqs), batch):
        lote = seqs[i:i + batch]
        enc = tok(lote, return_tensors='pt', padding=True)
        with torch.no_grad():
            hs = forward(enc)          # lista de tensores (B x L x d), índice 0 = embeddings
        for j in range(len(lote)):
            ids = enc['input_ids'][j].tolist()
            keep, spans = spans_nucleotidicos(tok, ids)
            for c in capas:
                h = hs[c][j][keep].float().cpu().numpy()
                m, v = resumir(h, spans)
                media[c].append(m); vent[c].append(v)
        if i % (batch * 50) == 0:
            print(f'{nombre}: {i}/{len(seqs)} ({time.time() - t0:.0f} s)')
    out = {}
    for c in capas:
        out[f'capa{c}_media'] = np.stack(media[c]); out[f'capa{c}_ventanas'] = np.stack(vent[c])
    np.savez_compressed(f'{CARPETA}/emb_{nombre}.npz', **{k: v.astype(np.float16) for k, v in out.items()})
    print(f'{nombre}: listo en {time.time() - t0:.0f} s')
    return out

### HyenaDNA

In [ ]:
nom = 'LongSafari/hyenadna-small-32k-seqlen-hf'
tok = AutoTokenizer.from_pretrained(nom, trust_remote_code=True)
mod = AutoModel.from_pretrained(nom, trust_remote_code=True).to(dev).eval()

def fw_hyena(enc):
    return mod(input_ids=enc['input_ids'].to(dev), output_hidden_states=True).hidden_states

n_capas = len(fw_hyena(tok(seqs[:1], return_tensors='pt'))) - 1
emb = {'hyenadna': extraer('hyenadna', tok, fw_hyena, n_capas, batch=64)}
del mod; torch.cuda.empty_cache()

### Nucleotide Transformer v2

In [ ]:
nom = 'InstaDeepAI/nucleotide-transformer-v2-100m-multi-species'
tok = AutoTokenizer.from_pretrained(nom, trust_remote_code=True)
mod = AutoModelForMaskedLM.from_pretrained(nom, trust_remote_code=True).to(dev).eval()

def fw_nt(enc):
    return mod(input_ids=enc['input_ids'].to(dev), attention_mask=enc['attention_mask'].to(dev),
               output_hidden_states=True).hidden_states

n_capas = len(fw_nt(tok(seqs[:1], return_tensors='pt'))) - 1
emb['nt_v2_100m'] = extraer('nt_v2_100m', tok, fw_nt, n_capas, batch=64)
del mod; torch.cuda.empty_cache()

### DNABERT-2

Su implementación propia no acepta `output_hidden_states`: las capas se piden con
`output_all_encoded_layers=True` y salen "despaddeadas" (tokens × d, sin dimensión de lote). Con lotes
de 1 secuencia eso es directamente la matriz de la secuencia. Tarda unos minutos más que los otros dos.

In [ ]:
from transformers.models.bert.configuration_bert import BertConfig
nom = 'zhihan1996/DNABERT-2-117M'
tok = AutoTokenizer.from_pretrained(nom, trust_remote_code=True)
cfg = BertConfig.from_pretrained(nom)
mod = AutoModel.from_pretrained(nom, trust_remote_code=True, config=cfg).to(dev).eval()
mod.pooler = None   # con output_all_encoded_layers la última capa sale sin dimensión de lote y el pooler falla

def fw_dnabert2(enc):
    capas, _ = mod(enc['input_ids'].to(dev), attention_mask=enc['attention_mask'].to(dev),
                   output_all_encoded_layers=True)
    capas = [c.reshape(1, -1, c.shape[-1]) for c in capas]
    return [None] + capas      # sin salida de embeddings: índice 1..12 = bloques

emb['dnabert2'] = extraer('dnabert2', tok, fw_dnabert2, cfg.num_hidden_layers, batch=1)
del mod; torch.cuda.empty_cache()

## Evaluación

Mismo split y mismo ridge que las bases. Para retomar sin volver a extraer: `emb[nombre] = dict(np.load(f'{CARPETA}/emb_{nombre}.npz'))`
(convertir a float32 con `{k: v.astype('float32') for k, v in ...}`).

In [ ]:
res, por_capa = [], []
for nombre, d in emb.items():
    for pooling in ('media', 'ventanas'):
        capas = {k.split('_')[0]: v.astype(np.float32) for k, v in d.items() if k.endswith(pooling)}
        t0 = time.time()
        a, b = ev.evaluar_embeddings(df, capas, f'{nombre}_{pooling}')
        res.append(a); por_capa.append(b)
        print(f'{nombre}_{pooling}: {time.time() - t0:.0f} s')
res = pd.concat(res); por_capa = pd.concat(por_capa)
res.to_csv(f'{CARPETA}/embeddings.tsv', sep='\t', index=False)
por_capa.to_csv(f'{CARPETA}/embeddings_por_capa.tsv', sep='\t', index=False)
res.round(3)

In [ ]:
# Comparación con las bases (subí también bases.tsv a la carpeta, o salteá esta celda)
import os
if os.path.exists(f'{CARPETA}/bases.tsv'):
    todo = pd.concat([pd.read_csv(f'{CARPETA}/bases.tsv', sep='\t'), res])
    display(todo.pivot_table(index='modelo', columns='replica', values=['r2', 'frac_techo', 'spearman'])
                .round(3).sort_values(('r2', 'Rep 1')))